In [3]:
#-------------------Task 1-------------------#


def manhattan(pos, goal):
    return abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])

def best_first_search(maze, start, goal):
    """Find a path from start to goal using Greedy Best-First Search (list-based frontier)."""
    rows, cols = len(maze), len(maze[0])

    frontier = [(start, manhattan(start, goal))]
    visited = set()
    came_from = {start: None}

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_pos, _ = frontier.pop(0)

        if current_pos in visited:
            continue

        visited.add(current_pos)

        if current_pos == goal:
            path = []
            while current_pos is not None:
                path.append(current_pos)
                current_pos = came_from[current_pos]
            path.reverse()
            return path

        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            new_pos = (current_pos[0] + dx, current_pos[1] + dy)
            if (0 <= new_pos[0] < rows and 0 <= new_pos[1] < cols and
                    maze[new_pos[0]][new_pos[1]] == 0 and new_pos not in visited):
                came_from[new_pos] = current_pos
                frontier.append((new_pos, manhattan(new_pos, goal)))

    return None

def get_permutations(items):
    if len(items) <= 1:
        return [list(items)]
    result = []
    for i in range(len(items)):
        rest = items[:i] + items[i+1:]
        for perm in get_permutations(rest):
            result.append([items[i]] + perm)
    return result

def find_multi_goal_path(maze, start, goals):
    best_path = None
    best_length = float('inf')
    best_order = None

    for goal_order in get_permutations(list(goals)):
        current_pos = start
        total_path = []
        valid = True

        for goal in goal_order:
            segment = best_first_search(maze, current_pos, goal)
            if segment is None:
                valid = False
                break
            total_path.extend(segment if not total_path else segment[1:])
            current_pos = goal

        if valid and len(total_path) < best_length:
            best_length = len(total_path)
            best_path = total_path
            best_order = goal_order

    if best_path:
        print(f"Optimal goal visit order: {best_order}")
    return best_path

def visualize_maze(maze, path, start, goals):
    visual = [['#' if cell == 1 else '.' for cell in row] for row in maze]
    for r, c in path:
        visual[r][c] = '*'
    visual[start[0]][start[1]] = 'S'
    for i, (r, c) in enumerate(goals):
        visual[r][c] = str(i + 1)
    print("\nMaze (S=start, 1/2/3=goals, *=path, #=wall, .=open):")
    for row in visual:
        print(' '.join(row))

maze = [
    [0, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [1, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0],
    [0, 1, 0, 0, 0, 0]
]

start = (0, 0)
goals = [(0, 5), (5, 0), (5, 5)]  # Three goal points spread across the maze

print("=== Enhanced Maze Navigation with Multiple Goals ===")
print(f"Start : {start}")
print(f"Goals : {goals}\n")

path = find_multi_goal_path(maze, start, goals)
if path:
    print(f"Path visiting all goals: {path}")
    print(f"Total steps: {len(path) - 1}")
    visualize_maze(maze, path, start, goals)
else:
    print("No path found that visits all goals")


=== Enhanced Maze Navigation with Multiple Goals ===
Start : (0, 0)
Goals : [(0, 5), (5, 0), (5, 5)]

Optimal goal visit order: [(0, 5), (5, 5), (5, 0)]
Path visiting all goals: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (3, 4), (3, 5), (2, 5), (1, 5), (0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 4), (5, 3), (5, 2), (4, 2), (4, 1), (4, 0), (5, 0)]
Total steps: 23

Maze (S=start, 1/2/3=goals, *=path, #=wall, .=open):
S . # . . 1
* # . # # *
* * * * # *
# # # * * *
* * * # # *
2 # * * * 3


In [4]:
#-------------------Task 2-------------------#

import random
import time

graph = {
    'A': {'B': 4, 'C': 3},
    'B': {'E': 12, 'F': 5},
    'C': {'D': 7, 'E': 10},
    'D': {'E': 2},
    'E': {'G': 5},
    'F': {'G': 16},
    'G': {}
}

heuristic = {'A': 14, 'B': 12, 'C': 11, 'D': 6, 'E': 4, 'F': 11, 'G': 0}


def update_edge_costs(graph):
    changed = {}
    for node in graph:
        for neighbor in graph[node]:
            old = graph[node][neighbor]
            graph[node][neighbor] = max(1, old + random.randint(-2, 2))
            if graph[node][neighbor] != old:
                changed[(node, neighbor)] = (old, graph[node][neighbor])
    return changed


def propagate_cost_change(node, g_costs, came_from, visited, frontier, graph, heuristic):
    queue = [node]
    while queue:
        parent = queue.pop()
        for neighbor in graph.get(parent, {}):
            if came_from.get(neighbor) == parent:
                new_g = g_costs[parent] + graph[parent][neighbor]
                if new_g != g_costs.get(neighbor):
                    g_costs[neighbor] = new_g
                    visited.discard(neighbor)
                    frontier.append((neighbor, new_g + heuristic[neighbor]))
                    queue.append(neighbor)


def dynamic_a_star(graph, start, goal, max_updates=3):
    print("\nFollowing is the Dynamic A* Search:\n")

    frontier  = [(start, heuristic[start])]
    visited   = set()
    g_costs   = {start: 0}
    came_from = {start: None}

    update_count   = 0
    step           = 0
    next_update_at = random.randint(2, 4)

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, current_f = frontier.pop(0)

        if current_node in visited:
            continue

        step += 1
        print(f"Step {step}: Expanding '{current_node}'  "
              f"(g={g_costs[current_node]}, h={heuristic[current_node]}, f={current_f})")
        visited.add(current_node)

        if current_node == goal:
            path = []
            node = current_node
            while node is not None:
                path.append(node)
                node = came_from[node]
            path.reverse()
            print(f"\nGoal reached!  Path: {' -> '.join(path)}  |  Total cost: {g_costs[goal]}")
            return path

        for neighbor, cost in graph[current_node].items():
            new_g = g_costs[current_node] + cost
            if neighbor not in g_costs or new_g < g_costs[neighbor]:
                g_costs[neighbor]   = new_g
                came_from[neighbor] = current_node
                frontier.append((neighbor, new_g + heuristic[neighbor]))

        if step >= next_update_at and update_count < max_updates:
            update_count += 1
            delay = random.uniform(0.3, 1.2)          # random real-time interval
            print(f"\n{'=' * 52}")
            print(f"  Dynamic Update #{update_count}  (triggered at step {step}, delay={delay:.2f}s)")
            time.sleep(delay)

            changed = update_edge_costs(graph)
            if changed:
                display = {f"{u}->{v}": f"{old}->{new}" for (u, v), (old, new) in changed.items()}
                print(f"  Edge changes: {display}")

                for (u, v), (old_cost, new_cost) in changed.items():
                    if u not in g_costs:
                        continue

                    new_g_v = g_costs[u] + new_cost

                    if new_cost < old_cost:
                        if new_g_v < g_costs.get(v, float('inf')):
                            print(f"  Improvement {u}->{v}: re-opening '{v}'  "
                                  f"(g: {g_costs.get(v)} -> {new_g_v})")
                            g_costs[v]   = new_g_v
                            came_from[v] = u
                            visited.discard(v)
                            frontier.append((v, new_g_v + heuristic[v]))
                            # Cascade the improvement to v's downstream tree
                            propagate_cost_change(v, g_costs, came_from, visited,
                                                  frontier, graph, heuristic)

                    else:
                        if came_from.get(v) == u:
                            print(f"  Degradation {u}->{v}: re-evaluating '{v}'  "
                                  f"(g: {g_costs.get(v)} -> {new_g_v})")
                            g_costs[v] = new_g_v
                            visited.discard(v)
                            frontier.append((v, new_g_v + heuristic[v]))
                            propagate_cost_change(v, g_costs, came_from, visited,
                                                  frontier, graph, heuristic)
            else:
                print("  No edges changed this interval.")

            next_update_at = step + random.randint(2, 4)
            print(f"  Resuming search  (next update around step {next_update_at})")
            print('=' * 52 + '\n')

    print("\nGoal not found")
    return None

dynamic_a_star(graph, 'A', 'G', max_updates=3)


Following is the Dynamic A* Search:

Step 1: Expanding 'A'  (g=0, h=14, f=14)
Step 2: Expanding 'C'  (g=3, h=11, f=14)
Step 3: Expanding 'B'  (g=4, h=12, f=16)

  Dynamic Update #1  (triggered at step 3, delay=0.84s)
  Edge changes: {'A->B': '4->3', 'B->E': '12->11', 'B->F': '5->3', 'C->D': '7->8', 'E->G': '5->4', 'F->G': '16->18'}
  Improvement A->B: re-opening 'B'  (g: 4 -> 3)
  Degradation C->D: re-evaluating 'D'  (g: 10 -> 11)
  Improvement E->G: re-opening 'G'  (g: None -> 17)
  Resuming search  (next update around step 6)

Step 4: Expanding 'B'  (g=3, h=12, f=15)
Step 5: Expanding 'D'  (g=11, h=6, f=16)
Step 6: Expanding 'E'  (g=13, h=4, f=17)

  Dynamic Update #2  (triggered at step 6, delay=0.77s)
  Edge changes: {'A->B': '3->1', 'A->C': '3->1', 'B->E': '11->9', 'C->D': '8->7', 'D->E': '2->1'}
  Improvement A->B: re-opening 'B'  (g: 3 -> 1)
  Improvement A->C: re-opening 'C'  (g: 3 -> 1)
  Improvement B->E: re-opening 'E'  (g: 11 -> 10)
  Improvement D->E: re-opening 'E'  (g: 

['A', 'C', 'D', 'E', 'G']

In [5]:
#-------------------Task 3-------------------#

graph = {
    'Warehouse': {'D1': 5, 'D2': 8, 'D3': 12},
    'D1':        {'D2': 4, 'D4': 7},
    'D2':        {'D3': 3, 'D5': 6},
    'D3':        {'D5': 5, 'D6': 4},
    'D4':        {'D6': 8},
    'D5':        {'D6': 3},
    'D6':        {}
}

time_windows = {
    'Warehouse': (0,  999),
    'D1':        (0,  20),   # Very strict
    'D2':        (5,  35),
    'D3':        (10, 40),
    'D4':        (0,  25),   # Strict
    'D5':        (15, 50),
    'D6':        (20, 60),   # Most flexible
}

heuristic = {node: tw[1] for node, tw in time_windows.items()}

def greedy_bfs(graph, start, goal):
    frontier = [(start, heuristic[start], 0)]
    visited = set()
    came_from = {start: None}

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, _, current_time = frontier.pop(0)

        if current_node in visited:
            continue

        visited.add(current_node)

        open_t, deadline = time_windows[current_node]
        if current_time > deadline:
            status = f"LATE by {current_time - deadline} min!"
        elif current_time < open_t:
            status = f"Too early, wait {open_t - current_time} min"
        else:
            status = "On time"
        print(f"{current_node} (t={current_time} min) [{open_t}-{deadline}] -> {status}")

        if current_node == goal:
            path = []
            node = current_node
            while node is not None:
                path.append(node)
                node = came_from[node]
            path.reverse()
            print(f"\nGoal found with GBFS. Path: {path}")
            print(f"Total travel time: {current_time} minutes")
            return

        for neighbor, travel_cost in graph[current_node].items():
            if neighbor not in visited:
                new_arrival = current_time + travel_cost
                came_from[neighbor] = current_node
                frontier.append((neighbor, heuristic[neighbor], new_arrival))

    print("\nGoal not found")

print("\nFollowing is the Greedy Best-First Search (GBFS):")
greedy_bfs(graph, 'Warehouse', 'D6')




Following is the Greedy Best-First Search (GBFS):
Warehouse (t=0 min) [0-999] -> On time
D1 (t=5 min) [0-20] -> On time
D4 (t=12 min) [0-25] -> On time
D2 (t=8 min) [5-35] -> On time
D3 (t=12 min) [10-40] -> On time
D5 (t=14 min) [15-50] -> Too early, wait 1 min
D6 (t=20 min) [20-60] -> On time

Goal found with GBFS. Path: ['Warehouse', 'D1', 'D2', 'D3', 'D5', 'D6']
Total travel time: 20 minutes
